# Polarization Rotation on the Healpix Sphere

All Healpix maps have `NSIDE=8192`, and all flat-sky maps have 2' resolution.

In [1]:
res = 2
interp = False

In [2]:
%pylab inline 

import numpy as np

from spt3g.core import G3Units

from spt3g.coordinateutils import FlatSkyMap, HealpixSkyMap
from spt3g.coordinateutils.fitsio import load_skymap_fits
from spt3g.coordinateutils.maputils import healpix_to_flatsky, get_ra_dec_map

from spt3g.std_processing.sourcemaps import CreateFieldMapStub
from spt3g.mapspectra.map_analysis import flatten_pol, constructEB, mapffts_to_powerspectra
from spt3g.util.files import load_pickle
from spt3g.mapspectra.map_spectrum_classes import MapSpectrum1DDict
from spt3g.lensing.map_spec_utils import get_camb_scalcl

import sys; sys.path.append("/src/spt3g_software/scratch/cailmdaley")
from plotting import plotFrame

Populating the interactive namespace from numpy and matplotlib


/src/.pyenv/versions/3.7.3/lib/python3.7/site-packages/IPython/core/magics/pylab.py:160: UserWarning: pylab import has clobbered these variables: ['interp']
`%matplotlib` prevents importing * from pylab and numpy
  "\n`%matplotlib` prevents importing * from pylab and numpy"


## Current Method: flat-sky `flatten_pol` 

Constructing a flat-sky B map from Healpix TQU goes something like this:

In [3]:
# mask = load_pickle("/spt/user/ddutcher/masks/3band_res2_bundle_apod_mask.pkl")
mask = np.zeros(CreateFieldMapStub(res=res*G3Units.arcmin).shape)
mask[50:-50, 50:-50] = 1

In [4]:
path = "/spt/simulation/lensing_maps/unlensed_TQU1/TQU1_planck2018_base_plikHM_TTTEEE_lowl_lowE_lensing_seed{}.fits"

# for seed in range(100, 105):
seed = 101
healpix_frame = load_skymap_fits(path.format(seed), memmap=True)
flatsky_frame = {k : healpix_to_flatsky(f, map_stub=CreateFieldMapStub(res=res*G3Units.arcmin), interp=interp) 
                  for (k,f) in healpix_frame.items()}

flatten_pol(flatsky_frame)
eb_frame = constructEB(flatsky_frame)
plotFrame(eb_frame, mult=mask, size=(14,4), suptitle="Seed {}".format(seed))

ArgumentError: Python argument types in
    spt3g.coordinateutils.reproj_map(HealpixSkyMap, FlatSkyMap)
did not match C++ signature:
    reproj_map(boost::shared_ptr<G3SkyMap const> in_map, boost::shared_ptr<G3SkyMap> out_map, int rebin=1, bool interp=False)

### Power Spectra

In [ ]:
spectra = mapffts_to_powerspectra(eb_frame, lmax=6000, delta_l=50, calculate_dls=True)
del spectra['EB']
plotFrame(spectra)

## Flattening on the sphere

In [ ]:
q, u = healpix_frame['Q'].Clone(True), healpix_frame['U'].Clone(True)
unrotated_flatsky_frame = {k : healpix_to_flatsky(f, map_stub=CreateFieldMapStub(res=res*G3Units.arcmin), interp=interp) 
                  for (k,f) in {'Q' : q, 'U': u}.items()}

map_stub=CreateFieldMapStub(res=res*G3Units.arcmin)
α0 = map_stub.alpha_center
δ0 = map_stub.delta_center

The polarization rotation angle depends on the derivatives of curved sky coordinates $\delta, \alpha$ with respect to flat-sky coordinates $x,y$, so we need to define a projection before we can do the rotation.

For the sake of computational thrift, we make a mask to select only the Healpix pixels that fall within our flat-sky map.

The derivatives are functions of $x$ and $y$, so we need to calculate those for our Healpix pixels.

In [ ]:
%%time 
(αmin, αmax), (δmin, δmax) = [[f(coord) for f in (np.min, np.max)] for coord in get_ra_dec_map(map_stub)]
α, δ  = get_ra_dec_map(q)

flatsky_mask = np.array([(αmin < α[i] < αmax) and (δmin < δ[i] < δmax) for i in range(len(α))])
α, δ = (np.asarray(healpix_coord)[flatsky_mask] for healpix_coord in (α, δ))

xy = np.array([map_stub.angle_to_xy(α[i], δ[i]) for i in range(len(α))])
stub_xy = np.array([map_stub.angle_to_xy(*map_stub.pixel_to_angle(i)) for i in range(len(map_stub))])
xy_mask = np.logical_and.reduce([bool_array for bool_set in [(stub_xy[:,i].min() <= xy[:,i], xy[:,i] <= stub_xy[:,i].max()) for i in range(2)] for bool_array in bool_set])

In [ ]:
# P = np.sqrt(np.asarray(q)[flatsky_mask]**2 + np.asarray(u)[flatsky_mask]**2)
# psi = np.arctan2(np.asarray(u)[flatsky_mask], np.asarray(q)[flatsky_mask]) / 2
# np.asarray(q)[flatsky_mask] = P
# np.asarray(u)[flatsky_mask] = psi

# Pflat, psiflat = [healpix_to_flatsky(f, map_stub=CreateFieldMapStub(res=res*G3Units.arcmin), interp=interp)
#                          for f in (q, u)]
# plotFrame({r"$P$" : np.asarray(Pflat)[0:100,0:100], r"$\psi$" : np.asarray(psiflat)[0:200,0:200]})
# flatsky_qu_frame = {'Q': Pflat.Clone(False), 'U': psiflat.Clone(False)}
# np.asarray(flatsky_qu_frame['Q'])[:] = Pflat * np.cos(2*psiflat)
# np.asarray(flatsky_qu_frame['U'])[:] = Pflat * np.sin(2*psiflat)
# plotFrame({k : np.asarray(f)[0:100,0:100] for (k,f) in flatsky_qu_frame.items()})
# flatten_pol(flatsky_qu_frame)
# plotFrame({k : np.asarray(f)[0:100,0:100] for (k,f) in flatsky_qu_frame.items()})
# eb_frame = constructEB(flatsky_qu_frame)
# plotFrame(eb_frame, mult=mask)

We can compare (the flat-sky subset of) the Healpix $x,y$ coordinates to the actual $x,y$ coordinates of our flat-sky map:

### Gradients

In [ ]:
def pixel_scale(xys, map_stub):
    ny, nx = map_stub.shape
    x0, y0 = map_stub.x_center, map_stub.y_center
    x_res, y_res = map_stub.x_res, map_stub.y_res
    
    x_min = (x0 - nx) * x_res
    y_min = (y0 - ny) * y_res
    
    xs = -1. * (xys[:,0] * x_res + x_min);
    ys = -1. * (xys[:,1] * y_res + y_min);
    return xs, ys

def grad_delta(x, y, d0):
    rho = np.sqrt(x**2 + y**2)
    h = np.sqrt(1 - rho**2 / 4)
    g = np.sqrt(1 - rho**2 / 2)

    denominator = rho**2 * h * np.sqrt(1 - (y*h*np.cos(d0) - g**2 * np.sin(d0))**2)
    c = (h**2 - g**2) * y * np.cos(d0) - rho**2 * h * np.sin(d0)
    
    grad_x =  x * c / denominator
    grad_y =   (y * c - (rho*h)**2 * np.cos(d0)) / denominator
    
    return grad_x, grad_y

def grad_alpha(x, y, d0):
    rho = np.sqrt(x**2 + y**2)
    h = np.sqrt(1 - (rho / 2)**2)
    g = np.sqrt(1 - rho**2 / 2)

    denominator = (x * rho * h)**2 + (rho * g**2 * np.cos(d0) + y * rho * h * np.sin(d0))**2
    c = np.cos(d0) * (rho**2 * h + g**4 / h - g**2 * h)
    
    grad_x = (x**2 * c + rho**2 * h * (g**2 * np.cos(d0) + y * h * np.sin(d0))) / denominator
    grad_y = x * (y * c - (rho*h)**2 * np.sin(d0)) / denominator
    
    return grad_x, grad_y

# compute gradients
δ_grd_x, δ_grd_y = grad_delta(*pixel_scale(xy, map_stub), δ0)
α_grd_x, α_grd_y = grad_alpha(*pixel_scale(xy, map_stub), δ0)

In [ ]:
inds = np.random.randint(np.sum(xy_mask), size=20000)
grads = {r"$\nabla_x \delta$": δ_grd_x,  r"$\nabla_x \alpha$": α_grd_x, 
         r"$\nabla_y \delta$": δ_grd_y,   r"$\nabla_y \alpha$": α_grd_y,
         r"$\phi_\delta$" : np.arctan2(δ_grd_y, δ_grd_x), r"$\phi_\alpha$" : np.arctan2(-α_grd_x, α_grd_y),
         r"$\phi_\delta - \phi_\alpha$" : np.arctan2(δ_grd_y, δ_grd_x) - np.arctan2(-α_grd_x, α_grd_y),
         r"$\phi_\delta - \phi_\alpha$ (again)" : np.arctan2(δ_grd_y, δ_grd_x) - np.arctan2(-α_grd_x, α_grd_y)}
fig = figure(figsize=(14, 16), tight_layout=True)
for i, k in enumerate(grads.keys()):
    subplot(421+i)
    scatter(xy[xy_mask][inds, 0], xy[xy_mask][inds, 1], c=grads[k][xy_mask][inds], s=25); 
    title(k); xlabel(r"$x$"); ylabel(r"$y$"); colorbar()

### Rotation Inspection

In [ ]:
def rotate_and_inspect(phi):
    q, u = healpix_frame['Q'].Clone(True), healpix_frame['U'].Clone(True)
    qu = -1*np.exp(1.0j * phi) * (np.asarray(q)[flatsky_mask]  + 1.0j * np.asarray(u)[flatsky_mask])
    np.asarray(q)[flatsky_mask] = np.real(qu)
    np.asarray(u)[flatsky_mask] = np.imag(qu)
    
    
    rotated_flatsky_frame = {k : healpix_to_flatsky(f, map_stub=CreateFieldMapStub(res=res*G3Units.arcmin), interp=interp)
                             for (k,f) in {'Q' : q, 'U': u}.items()}
    
    print("################################################################################:")
    print("QU Insets:")
    print("################################################################################:")
    pixmax = int(400 * 2 / res)
    plotFrame({k : np.asarray(f)[5:pixmax, 5:pixmax] for (k, f) in unrotated_flatsky_frame.items()}, suptitle="No rotation or flatten_pol", size=(10,4))
    plotFrame({k : np.asarray(f)[5:pixmax, 5:pixmax] for (k, f) in flatsky_frame.items() if k is not 'T'}, suptitle="Flatten_pol", size=(10,4))
    plotFrame({k : np.asarray(f)[5:pixmax, 5:pixmax] for (k, f) in rotated_flatsky_frame.items()}, suptitle="Rotation", size=(10,4))
    plotFrame({k : np.asarray(flatsky_frame[k] - rotated_flatsky_frame[k])[5:pixmax, 5:pixmax] for k in rotated_flatsky_frame.keys()}, suptitle="flatten_pol - rotation", size=(10,4))
    show()
    
    print("################################################################################:")
    print("EB maps:")
    print("################################################################################:")
    rotated_eb_frame = constructEB(rotated_flatsky_frame)
    unrotated_eb_frame = constructEB(unrotated_flatsky_frame)
    eb_frame = constructEB(flatsky_frame)
    
    plotFrame(unrotated_eb_frame, mult=mask, size=(14,4), suptitle="No rotation or flatten_pol")
    plotFrame(eb_frame, mult=mask, size=(14,4), suptitle="Flatten_pol")
    plotFrame(rotated_eb_frame, mult=mask, size=(14,4), suptitle="Rotated")
    plotFrame({k: eb_frame[k] - rotated_eb_frame[k] for k in eb_frame.keys()}, mult=mask, size=(14,4), suptitle="flatten_pol - rotated")
    show()
    
    print("################################################################################:")
    print("Power spectra:")
    print("################################################################################:")
    rotated_spectra = mapffts_to_powerspectra(rotated_eb_frame, lmax=6000, delta_l=50, calculate_dls=True)

    fig = figure(figsize=(10,4), tight_layout=True)
    for i, k in enumerate(['EE', 'BB']):
        subplot(121 + i)
        plot(spectra[k].bin_centers, spectra[k], label="flatten_pol")
        plot(rotated_spectra[k].bin_centers, rotated_spectra[k], label="rotated")
        ylabel(r"${}_\ell$".format(spectra[k].spec_type.upper()[0]))
        xlabel(r"$\ell$") 
        legend()
        

#### $\phi_\alpha + \phi_\delta$

In [ ]:
rotate_and_inspect(np.arctan2(-α_grd_x, α_grd_y) + np.arctan2(δ_grd_y, δ_grd_x))

#### $2\phi_\alpha$

In [ ]:
rotate_and_inspect(2*np.arctan2(-α_grd_x, α_grd_y))

#### $2\phi_\delta$

In [ ]:
rotate_and_inspect(2*np.arctan2(δ_grd_y, δ_grd_x))